# Wilcoxon Test and Decision Tree Baseline

This notebook completes two tasks for the paper's statistical analysis:

1. **Wilcoxon signed-rank test** (paired over the 10 folds) to support the claim "RF outperforms GB" with statistical significance
2. **Simple Decision Tree** (max depth 5) as an additional interpretable baseline

**Estimated time in Colab:** ~15 minutes (the decision tree is fast; most of the time goes into loading and preparing the data)

---
### Per-fold values already available (from PROCESO_MODELADO_CV_v2)

| Fold | RF-A F1 | GB-A F1 | RF-B F1 | GB-B F1 |
|------|---------|---------|---------|----------|
| 1    | 0.3817  | 0.3324  | 0.7704  | 0.7241   |
| 2    | 0.4128  | 0.3368  | 0.7735  | 0.7235   |
| 3    | 0.4051  | 0.3407  | 0.7661  | 0.7157   |
| 4    | 0.3810  | 0.3119  | 0.7749  | 0.7232   |
| 5    | 0.3954  | 0.3346  | 0.7715  | 0.7242   |
| 6    | 0.4046  | 0.3267  | 0.7694  | 0.7254   |
| 7    | 0.4124  | 0.3417  | 0.7727  | 0.7360   |
| 8    | 0.3892  | 0.3423  | 0.7682  | 0.7230   |
| 9    | 0.4006  | 0.3491  | 0.7693  | 0.7177   |
| 10   | 0.3775  | 0.3224  | 0.7699  | 0.7242   |

## Step 1 — Colab Setup

In [ ]:
from google.colab import drive
import os
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

# Install dependencies (scipy is already in Colab, but just in case)
!pip install scipy imbalanced-learn -q
# Step 3: copy the parquet file from Drive to the local Colab environment
# ─── ADJUST this path to the folder where you uploaded the parquet in your Drive ───
DRIVE_PARQUET = '/content/drive/MyDrive/fci-datos/part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet'
PARQUET = '/content/part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet'
# ────────────────────────────────────────────────────────────────────────────

if not os.path.exists(PARQUET):
    import shutil
    shutil.copy(DRIVE_PARQUET, PARQUET)
    print(f'Parquet copied to {PARQUET}')
else:
    print(f'Parquet already available at {PARQUET}')

print('Setup complete.')



Drive ya montado.
Parquet copiado a /content/part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet
Setup completado.


## Step 2 — Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from imblearn.over_sampling import RandomOverSampler
import warnings, time
warnings.filterwarnings('ignore')
print('Libraries loaded.')

Librerías cargadas.


## Step 3 — Wilcoxon Test (using values already available)

No need to retrain the models. The per-fold values from the previous notebook are entered directly.

In [ ]:
# ── F1-macro values per fold (from PROCESO_MODELADO_CV_v2_colab) ────────────
f1_rf_A  = [0.3817, 0.4128, 0.4051, 0.3810, 0.3954,
             0.4046, 0.4124, 0.3892, 0.4006, 0.3775]
f1_gb_A  = [0.3324, 0.3368, 0.3407, 0.3119, 0.3346,
             0.3267, 0.3417, 0.3423, 0.3491, 0.3224]

f1_rf_B  = [0.7704, 0.7735, 0.7661, 0.7749, 0.7715,
             0.7694, 0.7727, 0.7682, 0.7693, 0.7699]
f1_gb_B  = [0.7241, 0.7235, 0.7157, 0.7232, 0.7242,
             0.7254, 0.7360, 0.7230, 0.7177, 0.7242]

print('='*65)
print('WILCOXON SIGNED-RANK TEST (paired, k=10 folds)')
print('H0: no difference between RF and GB in F1-macro')
print('H1: RF > GB  (one-sided alternative)')
print('='*65)

for exp, rf_vals, gb_vals in [
    ('A (MAIN_ACTIVITY, 21 classes)', f1_rf_A, f1_gb_A),
    ('B (SME_WIN, binary)',         f1_rf_B, f1_gb_B),
]:
    rf = np.array(rf_vals)
    gb = np.array(gb_vals)
    diffs = rf - gb

    # Wilcoxon signed-rank, 'greater' alternative (RF > GB)
    stat, p_two   = wilcoxon(rf, gb, alternative='two-sided')
    stat, p_one   = wilcoxon(rf, gb, alternative='greater')

    print(f'\nExperiment {exp}')
    print(f'  RF  F1: {np.mean(rf):.4f} ± {np.std(rf):.4f}')
    print(f'  GB  F1: {np.mean(gb):.4f} ± {np.std(gb):.4f}')
    print(f'  Mean difference: {np.mean(diffs):+.4f}')
    print(f'  Wilcoxon statistic: {stat:.1f}')
    print(f'  p-value (two-sided): {p_two:.4f}')
    print(f'  p-value (RF > GB):   {p_one:.4f}')
    sig = '✓ SIGNIFICANT (p < 0.05)' if p_one < 0.05 else '✗ Not significant'
    print(f'  Result: {sig}')

print('\n--- LaTeX code to add as a note in the metrics table ---')
for exp, rf_vals, gb_vals, label in [
    ('A', f1_rf_A, f1_gb_A, 'Exp.~A'),
    ('B', f1_rf_B, f1_gb_B, 'Exp.~B'),
]:
    _, p = wilcoxon(np.array(rf_vals), np.array(gb_vals), alternative='greater')
    print(f'{label}: Wilcoxon signed-rank test, RF > GB, p = {p:.4f}')

TEST DE WILCOXON SIGNED-RANK (pareado, k=10 folds)
H0: no hay diferencia entre RF y GB en F1-macro
H1: RF > GB  (alternativa unilateral)

Experimento A (MAIN_ACTIVITY, 21 clases)
  RF  F1: 0.3960 ± 0.0124
  GB  F1: 0.3339 ± 0.0104
  Diferencia media: +0.0622
  Wilcoxon statistic: 55.0
  p-value (bilateral): 0.0020
  p-value (RF > GB):   0.0010
  Resultado: ✓ SIGNIFICATIVO (p < 0.05)

Experimento B (SME_WIN, binario)
  RF  F1: 0.7706 ± 0.0025
  GB  F1: 0.7237 ± 0.0051
  Diferencia media: +0.0469
  Wilcoxon statistic: 55.0
  p-value (bilateral): 0.0020
  p-value (RF > GB):   0.0010
  Resultado: ✓ SIGNIFICATIVO (p < 0.05)

--- CÓDIGO LaTeX para añadir como nota en la tabla de métricas ---
Exp.~A: Wilcoxon signed-rank test, RF > GB, p = 0.0010
Exp.~B: Wilcoxon signed-rank test, RF > GB, p = 0.0010


## Step 4 — Decision Tree as an Interpretable Baseline

Trained with the same k=10 protocol as RF and GB.
Max depth = 5 to keep it interpretable.

> **Adjust the parquet path** if needed.

In [ ]:
# ─── ADJUST THIS PATH ───────────────────────────────────────────────────
PARQUET = '/content/part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet'
# ────────────────────────────────────────────────────────────────────────

df = pd.read_parquet(PARQUET)
for col in df.select_dtypes(include='string').columns:
    df[col] = df[col].astype(object)
df['DURATION']   = df['DURATION'].astype(float)
df['VALUE_EURO'] = pd.to_numeric(df['VALUE_EURO'], errors='coerce').astype(float)
for col in ['NUMBER_AWARDS','LOTS_NUMBER','NUMBER_OFFERS','NUMBER_TENDERS_SME']:
    df[col] = df[col].astype(int)
df['GROUP_CPV'] = df['CPV'].astype(str).str[:2]
df['SME_WIN']   = df['B_CONTRACTOR_SME'].astype(str)\
                    .str.contains('y', case=False, na=False).astype(int)

cat_cols = ['B_MULTIPLE_CAE','B_ON_BEHALF','CAE_TYPE','MAIN_ACTIVITY',
            'ISO_COUNTRY_CODE','TYPE_OF_CONTRACT','GROUP_CPV']
for col in cat_cols: df[col] = df[col].astype(str)
ohe     = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_arr = ohe.fit_transform(df[cat_cols])
ohe_cols= ohe.get_feature_names_out(cat_cols)
num_cols= ['NUMBER_AWARDS','LOTS_NUMBER','NUMBER_OFFERS','NUMBER_TENDERS_SME']
df_proc = pd.concat([
    df[num_cols+['MAIN_ACTIVITY','SME_WIN']].reset_index(drop=True),
    pd.DataFrame(ohe_arr, columns=ohe_cols)
], axis=1)

uf_A = [c for c in [
    'B_MULTIPLE_CAE_n','B_ON_BEHALF_n','GROUP_CPV_45','GROUP_CPV_33','GROUP_CPV_15',
    'TYPE_OF_CONTRACT_w','MAIN_ACTIVITY_health','ISO_COUNTRY_CODE_si',
    'NUMBER_AWARDS','LOTS_NUMBER','NUMBER_OFFERS','NUMBER_TENDERS_SME',
    'CAE_TYPE_3','MAIN_ACTIVITY_general public\\services',
    'CAE_TYPE_4','CAE_TYPE_5','ISO_COUNTRY_CODE_lu'] if c in df_proc.columns]

uf_B = [c for c in [
    'B_MULTIPLE_CAE_n','B_ON_BEHALF_n','GROUP_CPV_45','GROUP_CPV_33','GROUP_CPV_15',
    'TYPE_OF_CONTRACT_w','MAIN_ACTIVITY_health','ISO_COUNTRY_CODE_si',
    'NUMBER_AWARDS','LOTS_NUMBER','NUMBER_OFFERS',
    'CAE_TYPE_3','MAIN_ACTIVITY_general public\\services',
    'CAE_TYPE_4','CAE_TYPE_5','ISO_COUNTRY_CODE_lu'] if c in df_proc.columns]

print(f'Dataset: {df_proc.shape[0]:,} rows')
print(f'Features A: {len(uf_A)} | Features B: {len(uf_B)}')

Dataset: 159,752 filas
Features A: 17 | Features B: 16


In [ ]:
def cv_arbol(X, y, exp_name, max_depth=5, k=10):
    """
    Validates a decision tree with the same k=10 inside-fold protocol.
    """
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    ros = RandomOverSampler(random_state=42)
    metricas = {'acc':[],'pre':[],'rec':[],'f1':[]}

    print(f'\n{"="*55}')
    print(f'Decision Tree (max_depth={max_depth}) — Experiment {exp_name}')
    print(f'{"="*55}')
    print(f'{"Fold":<6} {"Acc":>8} {"Pre":>8} {"Rec":>8} {"F1":>8}')
    print('-'*40)

    t0 = time.time()
    for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), 1):
        X_tr_b, y_tr_b = ros.fit_resample(X[tr_idx], y[tr_idx])
        dt = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
        dt.fit(X_tr_b, y_tr_b)
        yp = dt.predict(X[te_idx])

        acc = accuracy_score(y[te_idx], yp)
        pre = precision_score(y[te_idx], yp, average='macro', zero_division=0)
        rec = recall_score(y[te_idx], yp, average='macro', zero_division=0)
        f1  = f1_score(y[te_idx], yp, average='macro', zero_division=0)
        metricas['acc'].append(acc); metricas['pre'].append(pre)
        metricas['rec'].append(rec); metricas['f1'].append(f1)
        print(f'  {fold:<4} {acc:>8.4f} {pre:>8.4f} {rec:>8.4f} {f1:>8.4f}')

    print('-'*40)
    print(f'  Mean   {np.mean(metricas["acc"]):>8.4f} '
          f'{np.mean(metricas["pre"]):>8.4f} '
          f'{np.mean(metricas["rec"]):>8.4f} '
          f'{np.mean(metricas["f1"]):>8.4f}')
    print(f'  Std    {np.std(metricas["acc"]):>8.4f} '
          f'{np.std(metricas["pre"]):>8.4f} '
          f'{np.std(metricas["rec"]):>8.4f} '
          f'{np.std(metricas["f1"]):>8.4f}')
    print(f'  Total time: {(time.time()-t0)/60:.1f} min')
    return metricas

X_A  = df_proc[uf_A].astype(float).to_numpy()
y_A  = np.array(df_proc['MAIN_ACTIVITY'].astype(str).tolist())
X_B  = df_proc[uf_B].astype(float).to_numpy()
y_B  = df_proc['SME_WIN'].to_numpy()

# Experiment A (~3 min)
met_dt_A = cv_arbol(X_A, y_A, 'A (MAIN_ACTIVITY)')


Árbol de Decisión (max_depth=5) — Experimento A (MAIN_ACTIVITY)
Fold        Acc      Pre      Rec       F1
----------------------------------------
  1      0.5904   0.1762   0.2447   0.1447
  2      0.5884   0.1654   0.2626   0.1482
  3      0.5897   0.1675   0.2456   0.1498
  4      0.5916   0.1747   0.2476   0.1502
  5      0.5902   0.1688   0.2361   0.1461
  6      0.5915   0.1742   0.2515   0.1501
  7      0.5914   0.1747   0.2505   0.1480
  8      0.5926   0.1747   0.2577   0.1522
  9      0.5907   0.1729   0.2567   0.1468
  10     0.5921   0.1785   0.2592   0.1538
----------------------------------------
  Media    0.5909   0.1728   0.2512   0.1490
  Std      0.0012   0.0039   0.0076   0.0026
  Tiempo total: 1.1 min


In [ ]:
# Experiment B (~1 min)
met_dt_B = cv_arbol(X_B, y_B, 'B (SME_WIN)')


Árbol de Decisión (max_depth=5) — Experimento B (SME_WIN)
Fold        Acc      Pre      Rec       F1
----------------------------------------
  1      0.5888   0.5756   0.6268   0.5354
  2      0.5918   0.5779   0.6306   0.5384
  3      0.5783   0.5715   0.6201   0.5272
  4      0.5213   0.5735   0.6204   0.4923
  5      0.5833   0.5737   0.6237   0.5313
  6      0.5930   0.5799   0.6340   0.5402
  7      0.5316   0.5790   0.6300   0.5014
  8      0.5860   0.5756   0.6268   0.5338
  9      0.5780   0.5713   0.6198   0.5269
  10     0.5833   0.5709   0.6189   0.5296
----------------------------------------
  Media    0.5735   0.5749   0.6251   0.5256
  Std      0.0241   0.0031   0.0051   0.0151
  Tiempo total: 0.1 min


## Step 5 — Full Comparative Table and Wilcoxon vs Tree

In [ ]:
print('='*75)
print('FINAL COMPARATIVE TABLE (RF vs GB vs Decision Tree vs Baseline)')
print('='*75)

# F1 per fold from the previous models
f1_rf_A  = [0.3817,0.4128,0.4051,0.3810,0.3954,0.4046,0.4124,0.3892,0.4006,0.3775]
f1_gb_A  = [0.3324,0.3368,0.3407,0.3119,0.3346,0.3267,0.3417,0.3423,0.3491,0.3224]
f1_rf_B  = [0.7704,0.7735,0.7661,0.7749,0.7715,0.7694,0.7727,0.7682,0.7693,0.7699]
f1_gb_B  = [0.7241,0.7235,0.7157,0.7232,0.7242,0.7254,0.7360,0.7230,0.7177,0.7242]
f1_dt_A  = met_dt_A['f1']
f1_dt_B  = met_dt_B['f1']
baseline = 0.32  # majority-class classifier

print(f'\n{"Model":<25} {"Exp A F1":>16} {"Exp B F1":>16}')
print('-'*60)
for name, fa, fb in [
    ('Majority class (baseline)', [baseline]*10, None),
    ('Decision Tree (depth=5)',   f1_dt_A,  f1_dt_B),
    ('Gradient Boosting',        f1_gb_A,  f1_gb_B),
    ('Random Forest',            f1_rf_A,  f1_rf_B),
]:
    ma = f'{np.mean(fa):.4f} ± {np.std(fa):.4f}' if fa else 'N/A'
    mb = f'{np.mean(fb):.4f} ± {np.std(fb):.4f}' if fb else 'N/A'
    print(f'  {name:<23} {ma:>16} {mb:>16}')

print('\n--- WILCOXON TESTS (RF vs each baseline) ---')
for exp_name, rf_f1, others in [
    ('A', f1_rf_A, [('GB', f1_gb_A), ('DT', f1_dt_A)]),
    ('B', f1_rf_B, [('GB', f1_gb_B), ('DT', f1_dt_B)]),
]:
    print(f'\nExperiment {exp_name}:')
    for oname, of1 in others:
        _, p = wilcoxon(np.array(rf_f1), np.array(of1), alternative='greater')
        sig = '✓ p<0.05' if p < 0.05 else '✗ n.s.'
        print(f'  RF vs {oname}: p={p:.4f}  {sig}')

print('\n--- LaTeX CODE (footnote for the metrics table) ---')
for exp_name, rf_f1, gb_f1, dt_f1 in [
    ('A', f1_rf_A, f1_gb_A, f1_dt_A),
    ('B', f1_rf_B, f1_gb_B, f1_dt_B),
]:
    _, p_rf_gb = wilcoxon(np.array(rf_f1), np.array(gb_f1), alternative='greater')
    _, p_rf_dt = wilcoxon(np.array(rf_f1), np.array(dt_f1), alternative='greater')
    print(f'Exp {exp_name}: RF vs GB p={p_rf_gb:.4f}; RF vs DT p={p_rf_dt:.4f}')

TABLA COMPARATIVA FINAL (RF vs GB vs Árbol de Decisión vs Baseline)

Modelo                            Exp A F1         Exp B F1
------------------------------------------------------------
  Majority class (baseline)  0.3200 ± 0.0000              N/A
  Decision Tree (depth=5)  0.1490 ± 0.0026  0.5256 ± 0.0151
  Gradient Boosting        0.3339 ± 0.0104  0.7237 ± 0.0051
  Random Forest            0.3960 ± 0.0124  0.7706 ± 0.0025

--- TESTS DE WILCOXON (RF vs cada baseline) ---

Experimento A:
  RF vs GB: p=0.0010  ✓ p<0.05
  RF vs DT: p=0.0010  ✓ p<0.05

Experimento B:
  RF vs GB: p=0.0010  ✓ p<0.05
  RF vs DT: p=0.0010  ✓ p<0.05

--- CÓDIGO LaTeX (nota al pie tabla de métricas) ---
Exp A: RF vs GB p=0.0010; RF vs DT p=0.0010
Exp B: RF vs GB p=0.0010; RF vs DT p=0.0010


## Step 6 — Save Results to CSV

In [ ]:
rows = []
for modelo, exp, f1_vals in [
    ('RF',  'A', f1_rf_A), ('GB',  'A', f1_gb_A), ('DT', 'A', f1_dt_A),
    ('RF',  'B', f1_rf_B), ('GB',  'B', f1_gb_B), ('DT', 'B', f1_dt_B),
]:
    for i, v in enumerate(f1_vals, 1):
        rows.append({'modelo': modelo, 'exp': exp, 'fold': i, 'f1': v})

df_res = pd.DataFrame(rows)
df_res.to_csv('resultados_estadisticos.csv', index=False)
print('Saved: resultados_estadisticos.csv')
print(df_res.groupby(['exp','modelo'])['f1'].agg(['mean','std']).round(4))

Guardado: resultados_estadisticos.csv
              mean     std
exp modelo                
A   DT      0.1490  0.0028
    GB      0.3339  0.0110
    RF      0.3960  0.0131
B   DT      0.5256  0.0159
    GB      0.7237  0.0053
    RF      0.7706  0.0026
